# Active-Bias Retraining On Muller-Brown

This notebook runs the falsifiable follow-up to `13_adaptive_retraining_colab.ipynb`. The old mdshare replay is still useful as a numerical sanity check, but the stream is stationary. Here the bias is active: `ActiveBiasLedger` returns a force that enters a Langevin BAOAB integrator on the analytic Muller-Brown potential.

The implementation lives in `example_programs/14_muller_brown_active_bias.py` so it can be tested locally and reused from Colab without installing the whole package.

## 1. Colab Setup

Run this first. It avoids Poetry because this notebook only needs the scientific Python stack and Colab's global environment is fragile when Poetry tries to downgrade preinstalled packages.

In [ ]:
from pathlib import Path
import sys

IN_COLAB = "google.colab" in sys.modules

if IN_COLAB and not Path("pyproject.toml").exists():
    !git clone https://github.com/Komputerowe-Projektowanie-Lekow/pmarlo.git
    %cd pmarlo

if IN_COLAB:
    %pip install -q numpy pandas matplotlib scipy scikit-learn
    print("Colab dependencies installed for the active-bias Muller-Brown notebook.")
else:
    print("Local run detected. Use: poetry run jupyter notebook example_programs/14_muller_brown_active_bias_colab.ipynb")

## 2. Smoke Run

This runs a small grid to verify the active feedback loop, file outputs, plots, and aggregation. It uses `HILL_HEIGHT=2.0` because a 6000-step smoke run needs a stronger bias budget to leave the initial basin. Keep this as the first Colab run; do not use it as the final strategy ranking.

In [ ]:
%run example_programs/14_muller_brown_active_bias.py --quick --replicates 1 --budget-frames 3000 --hill-height 2.0 --output-dir example_programs/programs_outputs/muller_brown_active_bias_colab_smoke

## 3. Main Quick Analysis

This keeps the condition grid Colab-friendly while still testing active feedback. It is still a calibration run, not the final 3 x 4 x 3 protocol. Quick-mode training labels are intentionally `Fixed-10ep` and `EarlyStopping-20ep` so they are not confused with the full protocol. Increase replicas and use the full protocol after the smoke run is stable.

In [ ]:
%run example_programs/14_muller_brown_active_bias.py --quick --replicates 2 --budget-frames 6000 --hill-height 2.0 --on-retrain-policy reset_ledger --output-dir example_programs/programs_outputs/muller_brown_active_bias_colab_reset

## 4. Optional Full Protocol

Run this only after the quick analysis works. It uses the full 3 x 4 x 3 condition grid with 25000 steps, 10 replicas, and `HILL_HEIGHT=1.0`, giving about 50 units of deposited bias energy for a barrier of roughly 38 units.

In [ ]:
%run example_programs/14_muller_brown_active_bias.py --no-quick --replicates 10 --budget-frames 25000 --hill-height 1.0 --on-retrain-policy reset_ledger --output-dir example_programs/programs_outputs/muller_brown_active_bias_colab_reset_full

## 5. Quick Reproject Calibration

`reset_ledger` is easier to interpret and is the default. This quick `reproject_centers` run checks whether preserving old hills is numerically stable before the full protocol. The full reproject run is the next section.

In [ ]:
%run example_programs/14_muller_brown_active_bias.py --quick --replicates 2 --budget-frames 6000 --hill-height 2.0 --on-retrain-policy reproject_centers --output-dir example_programs/programs_outputs/muller_brown_active_bias_colab_reproject

## 6. Full Reproject Protocol

Run this after the full reset-ledger protocol. It uses the same 3 x 4 x 3 grid, 10 replicas, 25000 steps, and `HILL_HEIGHT=1.0`, but preserves old hills by storing their original `(x, y)` locations and projecting them into each retrained CV model.


In [ ]:
%run example_programs/14_muller_brown_active_bias.py --no-quick --replicates 10 --budget-frames 25000 --hill-height 1.0 --on-retrain-policy reproject_centers --output-dir example_programs/programs_outputs/muller_brown_active_bias_colab_reproject_full

## Outputs

Results are written to the explicit output directory in each `%run` command. For the hypothesis test, compare `muller_brown_active_bias_colab_reset_full` against `muller_brown_active_bias_colab_reproject_full`:

- `muller_brown_active_bias_results.csv`
- `muller_brown_active_bias_summary.csv`
- `muller_brown_active_bias_plots.png`
- `muller_brown_active_bias_protocol.json`

Use the Muller-Brown ranking for the hypothesis test. The older mdshare replay should be treated only as a sanity check for scoring and aggregation.